# Диффузионные модели

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Диффузионные модели».

## Научная цель

Диффузионная модель учится генерировать данные через постепенное «очищение шума». Сначала к настоящим объектам понемногу добавляют случайный шум, пока не остаётся чистый шум. Модель учат обращать этот процесс — шаг за шагом убирать шум. После обучения она стартует с чистого шума и за серию шагов превращает его в новый правдоподобный объект.

В этом блокноте мы реализуем диффузионную модель (схема DDPM) на компактных двумерных данных. Двумерность позволяет наглядно увидеть и прямой процесс зашумления, и обратную генерацию. Блокнот исполняется на CPU за минуту.

**Важное предупреждение.** Сгенерированные объекты правдоподобны, но не гарантированно физически или химически корректны — нужна проверка.

## Интуиция за методом

Представьте проявку фотографии в тёмной комнате: сначала виден лишь бесформенный серый лист, и с каждым шагом проявки всё чётче проступают детали, пока не сложится готовое изображение. Диффузионная модель работает по той же логике — но в обратном направлении.

Процесс обучения состоит из двух фаз:

1. **Прямой процесс (forward process)** — к настоящим объектам шаг за шагом добавляют случайный гауссов шум, пока объект полностью не «растворится» в шуме. Этот процесс не требует обучения — он задан математически.

2. **Обратный процесс (reverse process)** — нейронную сеть учат предсказывать, какой именно шум был добавлен на данном шаге. По сути, сеть учится «стирать ластиком» один шаг зашумления.

После обучения генерация выглядит так: берём чистый случайный шум и применяем сеть `T` раз подряд. На каждом шаге она убирает немного шума — и постепенно из бесформенного облака проступает структура.

Преимущество перед GAN: нет нестабильного соревнования двух сетей. Преимущество перед VAE: более высокое качество выборок. Цена — медленная генерация (нужны сотни шагов).

Терминологический минимум для этого блокнота:
- **Расписание шума** (noise schedule) — как быстро нарастает шум по шагам. Задаётся коэффициентами `beta`.
- **alpha_bar** — накопленное произведение коэффициентов: позволяет получить зашумлённый объект для любого шага сразу, без итераций.
- **SiLU** (Sigmoid Linear Unit) — функция активации: плавнее ReLU, хорошо работает в моделях предсказания шума.
- **Синусоидальное кодирование** — способ передать сети номер шага в виде числового вектора: разные частоты синусоид кодируют разные «масштабы» времени.
- **DDPM** (Denoising Diffusion Probabilistic Model) — классическая схема обучения и генерации, реализованная в этом блокноте.
- **Функция потерь MSE** — среднеквадратичная ошибка между предсказанным и настоящим шумом.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch scikit-learn numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем двумерный набор `make_moons` — два переплетённых полумесяца. На таком наборе видно и как данные постепенно тонут в шуме (прямой процесс), и как модель восстанавливает их форму из шума (обратный процесс).

In [ ]:
import numpy as np
import torch
from sklearn.datasets import make_moons

torch.manual_seed(0)
np.random.seed(0)

X, _ = make_moons(n_samples=3000, noise=0.06, random_state=0)
X = X.astype('float32')
X = (X - X.mean(0)) / X.std(0)
data = torch.from_numpy(X)
print(f'Точек в наборе: {len(X)}')

### Прямой процесс: расписание зашумления

Зададим расписание шума на `T = 200` шагов. На каждом шаге к данным добавляется немного гауссова шума.

Ключевой трюк: вместо последовательного применения шагов можно вычислить зашумлённый объект на любом шаге `t` сразу через накопленный коэффициент `alpha_bar[t]`. Формула: `x_t = sqrt(alpha_bar[t]) * x_0 + sqrt(1 - alpha_bar[t]) * epsilon`, где `epsilon` — стандартный нормальный шум.

При `alpha_bar` близком к 1 (ранние шаги) сигнал ещё хорошо виден. При `alpha_bar` близком к 0 (поздние шаги) остался почти только шум.

На графике ниже видно, как данные «тонут» в шуме по мере роста номера шага.

In [ ]:
T = 200                                  # число шагов диффузии
betas = torch.linspace(1e-4, 0.02, T)    # расписание шума
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0) # накопленный коэффициент


def q_sample(x0, t, noise):
    """Прямой процесс: зашумлённый объект на шаге t."""
    a = alpha_bar[t].unsqueeze(-1)
    return torch.sqrt(a) * x0 + torch.sqrt(1 - a) * noise


# Иллюстрация прямого процесса: данные постепенно тонут в шуме.
import matplotlib.pyplot as plt
show_steps = [0, 50, 120, 199]
fig, axes = plt.subplots(1, len(show_steps), figsize=(13.5, 3.8),
                         sharex=True, sharey=True)
for ax, st in zip(axes, show_steps):
    tt = torch.full((len(data),), st, dtype=torch.long)
    noised = q_sample(data, tt, torch.randn_like(data)).numpy()
    ax.scatter(noised[:, 0], noised[:, 1], s=6,
               color=VIZ['series'][0], alpha=0.4)
    ax.set_title(f'Шаг зашумления {st}')
    ax.set_xlabel('Признак 1')
axes[0].set_ylabel('Признак 2')
fig.suptitle('Прямой процесс: данные постепенно превращаются в шум',
             fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график (прямой процесс):**

Четыре панели показывают облако точек (два полумесяца) на разных шагах зашумления:
- **Шаг 0:** исходные данные — чёткие два полумесяца.
- **Шаг 50:** форма ещё угадывается, но точки «расплылись».
- **Шаг 120:** контуры почти стёрты — остался лишь намёк на структуру.
- **Шаг 199:** однородное гауссово облако — структура данных полностью поглощена шумом.

Именно этот процесс модель учится обращать.

## 4. Применение метода

### Шаг 1. Сеть-предсказатель шума

Сеть получает на вход:
1. Зашумлённый объект `x_t` (двумерная точка).
2. Номер шага `t` (целое число от 0 до T).

И должна вернуть оценку шума `epsilon`, который был добавлен к объекту.

Номер шага `t` нельзя подать напрямую как одно число — сети трудно «чувствовать» масштаб. Поэтому применяют **синусоидальное кодирование**: `t` преобразуется в вектор из синусов и косинусов разных частот. Это тот же трюк, что и в трансформерах для позиционного кодирования.

`SiLU` (Sigmoid Linear Unit) — функция активации `x * sigmoid(x)`: более гладкая альтернатива ReLU, часто дающая лучшее качество в диффузионных моделях.

In [ ]:
import torch.nn as nn


def time_embedding(t, dim=16):
    """Синусоидальное кодирование номера шага диффузии."""
    half = dim // 2
    freqs = torch.exp(-np.log(10000) *
                      torch.arange(half, dtype=torch.float32) / half)
    args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class NoisePredictor(nn.Module):
    """Предсказывает шум, добавленный к двумерному объекту."""

    def __init__(self, t_dim=16, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )
        self.t_dim = t_dim

    def forward(self, x, t):
        te = time_embedding(t, self.t_dim)
        return self.net(torch.cat([x, te], dim=-1))


model = NoisePredictor()
print(model)

### Шаг 2. Обучение сети

На каждом шаге обучения:
1. Берём случайный пакет объектов из данных (`x_0`).
2. Выбираем случайный шаг зашумления `t` для каждого объекта.
3. Генерируем случайный шум `epsilon`.
4. Вычисляем зашумлённый объект `x_t` через формулу прямого процесса.
5. Подаём `x_t` и `t` в сеть — она предсказывает `epsilon_pred`.
6. Считаем MSE между `epsilon_pred` и настоящим `epsilon`, обновляем веса.

Это обучение без разметки: метки — это шум, который мы сами добавили.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
crit = nn.MSELoss()

BATCH = 256
n_steps = 3000
history = []
for step in range(1, n_steps + 1):
    idx = torch.randint(0, len(data), (BATCH,))
    x0 = data[idx]
    t = torch.randint(0, T, (BATCH,))
    noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    # Сеть учится предсказывать добавленный шум.
    pred = model(xt, t)
    loss = crit(pred, noise)
    opt.zero_grad(); loss.backward(); opt.step()
    history.append(loss.item())
    if step % 600 == 0:
        recent = np.mean(history[-200:])
        print(f'Шаг {step:4d}: средняя потеря {recent:.4f}')

### Шаг 3. Обратный процесс: генерация из шума

Генерация — это итеративное удаление шума по схеме DDPM. Начинаем с чистого гауссова шума и применяем сеть `T = 200` раз в обратном порядке: от шага 199 до 0.

На каждом шаге `t`:
1. Сеть предсказывает шум `eps` в точке `x_t`.
2. Вычисляем «очищенное» среднее `mean` — оценку объекта без шума.
3. Если шаг ещё не финальный — добавляем немного нового случайного шума (это делает генерацию разнообразной).
4. Переходим к шагу `t-1`.

В снимки `snaps` сохраняем промежуточные состояния на разных шагах генерации.

In [ ]:
@torch.no_grad()
def sample(n, keep=(199, 120, 60, 0)):
    """Генерация n объектов; возвращает снимки на шагах keep."""
    x = torch.randn(n, 2)              # старт: чистый шум
    snaps = {}
    for step in reversed(range(T)):
        t = torch.full((n,), step, dtype=torch.long)
        eps = model(x, t)
        a = alphas[step]
        a_bar = alpha_bar[step]
        # Шаг обратного процесса DDPM.
        coef = (1 - a) / torch.sqrt(1 - a_bar)
        mean = (x - coef * eps) / torch.sqrt(a)
        if step > 0:
            x = mean + torch.sqrt(betas[step]) * torch.randn_like(x)
        else:
            x = mean
        if step in keep:
            snaps[step] = x.clone().numpy()
    return snaps


snaps = sample(800)
print('Генерация завершена.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

axes[0].plot(np.convolve(history, np.ones(50) / 50, mode='valid'),
             color=VIZ['series'][0])
axes[0].set_title('Обучение: ошибка предсказания шума')
axes[0].set_xlabel('Шаг обучения')
axes[0].set_ylabel('Среднеквадратичная потеря (сглажено)')

axes[1].scatter(X[:, 0], X[:, 1], s=10, color=VIZ['series'][3],
                alpha=0.3, label='Настоящие данные')
gen = snaps[0]
axes[1].scatter(gen[:, 0], gen[:, 1], s=12, color=VIZ['series'][1],
                alpha=0.7, label='Генерация диффузии')
axes[1].set_title('Итог генерации')
axes[1].set_xlabel('Признак 1')
axes[1].set_ylabel('Признак 2')
axes[1].legend(loc='upper right')

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Левый — кривая обучения:** ошибка предсказания шума (MSE) убывает — сеть учится точнее угадывать, какой шум был добавлен на любом уровне зашумления. Чем ниже кривая к концу, тем точнее будет генерация.
- **Правый — итог генерации:** серые точки — настоящие данные, цветные — сгенерированные. При хорошей модели два облака совпадают по форме и плотности. Оба полумесяца должны быть воспроизведены.

In [ ]:
# Обратный процесс: как шум постепенно превращается в данные.
order = sorted(snaps, reverse=True)
fig, axes = plt.subplots(1, len(order), figsize=(13.5, 3.8),
                         sharex=True, sharey=True)
for ax, st in zip(axes, order):
    pts = snaps[st]
    ax.scatter(pts[:, 0], pts[:, 1], s=7, color=VIZ['series'][1],
               alpha=0.55)
    ax.set_title(f'Шаг очистки {T - st}')
    ax.set_xlabel('Признак 1')
axes[0].set_ylabel('Признак 2')
fig.suptitle('Обратный процесс: из шума постепенно проступает форма данных',
             fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график (обратный процесс):**

Четыре панели показывают промежуточные состояния генерации — от чистого шума до финального результата:
- **Шаг очистки 1 (самый левый)** — стартовое облако из чистого гауссова шума.
- **Шаги 80–140** — постепенно проступает форма: сначала намёк на изгиб, потом два отдельных скопления.
- **Шаг 200 (самый правый)** — финальный результат: два полумесяца воспроизведены из шума.

«Шаг очистки N» означает, что модель уже сделала N шагов обратного процесса (убрала шум N раз). Чем правее — тем чище результат.

## 5. Интерпретация результата

- **Прямой процесс** (раздел 3): данные постепенно тонут в шуме. Это та траектория, которую модель учится обращать.
- **Кривая обучения**: ошибка предсказания шума снижается — сеть учится распознавать добавленный шум на любом уровне зашумления.
- **Обратный процесс**: из бесформенного облака шума шаг за шагом проступает структура двух полумесяцев. Это и есть генерация.
- **Итог генерации**: облако сгенерированных точек должно совпадать с настоящими данными по форме и плотности.

Диффузионные модели обучаются стабильнее состязательных и дают качественные разнообразные выборки, но генерация требует многих шагов и потому медленнее. Сгенерированные объекты правдоподобны, но их корректность нужно проверять независимо.

## 6. Попробуйте на своих данных

Эта учебная реализация работает с векторными данными невысокой размерности.

1. Подготовьте матрицу `X` формы `(наблюдение, признак)` и нормируйте признаки (нулевое среднее, единичный разброс).
2. Снимите комментарии в ячейке ниже и подставьте свои данные; при иной размерности замените число `2` на число признаков в `NoisePredictor`.
3. Число шагов `T` и расписание `betas` влияют на качество и скорость; для ускорения генерации применяют сэмплер DDIM.
4. Для изображений берут свёрточную сеть-предсказатель шума (архитектура U-Net).

## 7. Поэкспериментируйте сами

Измените один параметр и перезапустите соответствующие ячейки:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `T = 100` | Вдвое меньше шагов диффузии | Меньше шагов = быстрее генерация. Хуже ли качество? |
| `T = 400` | Вдвое больше шагов | Улучшается ли воспроизводство формы данных? |
| `betas` — линейное расписание | Заменить на `torch.linspace(1e-4, 0.05, T)` (более крутой спад) | Данные тонут в шуме быстрее — успевает ли сеть обучиться? |
| `n_steps = 1500` | Меньше шагов обучения | Как убывает кривая? Насколько хуже генерация? |
| `hidden = 64` | Уменьшить сеть | Достаточно ли малой сети для 2D-данных? |

Дополнительный эксперимент: в функции `sample` уберите добавление шума (закомментируйте строку с `torch.randn_like(x)`) и используйте только `mean`. Что происходит с разнообразием сгенерированных точек?

In [ ]:
# --- Шаблон загрузки своих векторных данных ---
# import pandas as pd
#
# X = pd.read_csv('my_data.csv').to_numpy(dtype='float32')
# X = (X - X.mean(0)) / (X.std(0) + 1e-8)
# data = torch.from_numpy(X)
#
# n_features = X.shape[1]
# print(f'Объектов: {len(X)}, признаков: {n_features}')
# В NoisePredictor замените размерность 2 на n_features.

## 8. Краткие выводы

- Диффузионная модель обучается через простую задачу предсказания шума: сеть угадывает, какой шум добавили к объекту на каждом шаге зашумления.
- Прямой процесс — математически заданное постепенное зашумление. Обратный — нейронная сеть, применяемая итеративно.
- Генерация требует T последовательных шагов — это медленнее GAN, но стабильнее в обучении и обычно даёт более качественные и разнообразные выборки.
- В 2D можно буквально видеть, как из облака шума «проявляется» форма данных — это и есть суть диффузии.
- Сгенерированные объекты правдоподобны, но их физическую или химическую корректность нужно проверять независимо.
- Для реальных задач берут U-Net как предсказатель шума для изображений или эквивариантные сети для молекул.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике обратного процесса при шаге очистки 80 форма двух полумесяцев уже угадывается, но точки ещё сильно размыты. На шаге 200 структура чёткая. Что именно делает сеть между этими двумя шагами и почему нельзя просто пропустить промежуточные шаги?
2. Вы уменьшили число шагов диффузии с `T=200` до `T=50` и генерация стала заметно быстрее, но сгенерированные точки плохо воспроизводят форму полумесяцев. Какова физическая причина ухудшения качества и какой подход позволяет ускорить генерацию без столь сильной потери качества?
3. Ошибка предсказания шума на кривой обучения перестала убывать после 1500 шагов, но при визуальном осмотре сгенерированные точки не образуют чётких полумесяцев. Назовите два наиболее вероятных объяснения этого расхождения.

<details>
<summary>Показать ориентиры для ответов</summary>

1. На каждом шаге обратного процесса сеть убирает ровно столько шума, на сколько она обучена: она предсказывает шум конкретного уровня зашумления `t` и корректирует точку на один небольшой шаг. Пропуск шагов нарушает распределение, к которому обучена сеть: она ожидает получить `x_t`, а получает `x_{t-k}` с иным уровнем шума. Результат — накопленная ошибка и артефакты.
2. При малом `T` каждый шаг диффузии добавляет слишком много шума за раз (большие `beta`), и предположение о гауссовом приближении обратного шага перестаёт быть точным. Для ускорения без потери качества применяют детерминированный сэмплер DDIM (Denoising Diffusion Implicit Models): он позволяет делать большие шаги, не нарушая гауссово приближение, сокращая число шагов генерации в 10–50 раз.
3. Два вероятных объяснения: (а) модель достигла локального минимума при недостаточной ёмкости сети — функция потерь стабилизировалась, но сеть не выучила тонкую структуру данных; нужно увеличить `hidden` или число слоёв. (б) Данные нормированы неправильно или расписание шума `betas` не покрывает нужный диапазон — финальное зашумлённое распределение не является стандартным нормальным, и обратный процесс не возвращает к исходному распределению.
</details>
